In [ ]:
import os
os.chdir(os.pardir)

from src.utils import AIDatasetLoader, filter_by_app_and_model, DecisionTreeInterpreter, LogisticRegressionInterpreter  
from src.memory import Chunk, DeclarativeMemory

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## 1. Load datasets

In [ ]:
current_dir = os.getcwd()
data_dir = os.path.join(current_dir, 'datasets')

file_values = os.path.join(data_dir, 'values.csv')
file_metadata = os.path.join(data_dir, 'metadata.csv')
file_prediction = os.path.join(data_dir, 'none.csv')

values_df = pd.read_csv(file_values)
metadata_df = pd.read_csv(file_metadata)
prediction_df = pd.read_csv(file_prediction)

# ✅ Which model to use per dataset
dataset_model_map = {
    "mushrooms": "mlp",
    "wine_quality": "mlp",
    "forest_cover": "xgboost",
    "adult": "xgboost",
}

# # 🔧 Choose dataset here:
# app_id = "wine_quality"  # 🔄 Change this line to switch datasets

# # 🧠 Auto-configured values:
# model_name = dataset_model_map[app_id]

# load ai loader
ai_dataset_loader = AIDatasetLoader(
    feature_values_df=values_df,
    metadata_df=metadata_df,
    AI_predictions_df=prediction_df
)

# Load decision tree
dt_df = pd.read_csv(os.path.join(data_dir, 'decision_tree.csv'))
# dt_exp = DecisionTreeInterpreter(dt_df, metadata_df, app_id, model_name, depth=2)
# dt_exp.print_tree(as_name=True)


# Load linear model
lr_df = pd.read_csv(os.path.join(data_dir, 'logistic_regression.csv'))
# lr_exp = LogisticRegressionInterpreter(lr_df, metadata_df, app_id, model_name, variant="sparse")

In [8]:

import importlib
import src.memory as memory
importlib.reload(memory)
import src.dt_memory as dt_memory
importlib.reload(dt_memory)
import src.heuristic_lr_model as heuristic_lr_model
importlib.reload(heuristic_lr_model)
import src.lr_memory as lr_memory
importlib.reload(lr_memory)

from src.memory import DeclarativeMemory, CombinedMemory
from src.dt_memory import (
    add_dt_to_memory, dt_traverse, refresh_dt_path_in_memory
)
from src.heuristic_lr_model import (
    add_lr_heuristic_to_memory, lr_heuristic, refresh_lr_heuristic_in_memory
)
from src.lr_memory import (
    add_lr_calculation_to_memory, lr_calculation, refresh_lr_calculation_in_memory
)

from typing import Optional
import random
from dataclasses import replace

In [ ]:
# ---------------- Memory builder ----------------
def _make_memory(retrieval_threshold, latency_factor):
    dm = DeclarativeMemory(
        retrieval_threshold=retrieval_threshold,
        latency_factor=latency_factor,
        latency_exponent=0.5,
        max_assoc_strength=2.0,
        mismatch_penalty=-1.0,
        activation_noise=0.3,
        decay=0.5,
    )
    return CombinedMemory(dm, wm_capacity=7)

def _select_prob(probs, y):
    if hasattr(probs, "__len__") and len(probs) == 2:
        return float(probs[1]) if float(y) > 0 else float(probs[0])
    raise ValueError(f"Unexpected probs format: {probs}")


# ---------------- Strategy adapters ----------------
def _seed_memory_for_strategy(strategy, memory, explainer, compute_sf):
    """
    Initializes memory for the chosen strategy.
    """
    if strategy == "lr_calc":
        # preload LR coefficients for calculation path
        add_lr_calculation_to_memory(explainer, memory)
    elif strategy == "lr_heur":
        # preload LR heuristic state (use your preferred init var)
        add_lr_heuristic_to_memory(explainer, memory, initial_var=1.0)
    elif strategy == "dt":
        # preload DT structure (threshold sig figs follow compute_sf)
        add_dt_to_memory(memory, explainer, thresh_sf=int(compute_sf))
    else:
        raise ValueError(f"Unknown strategy: {strategy!r}")


def _predict_once(strategy, instance, norm_instance, memory, explainer, *, with_xai,
                  T_enc, T_op, ddm_a, ddm_s, compute_sf):
    """
    Runs ONE trial under the chosen strategy and returns:
      probs, pred_time, aux_info
    """
    if strategy == "lr_calc":
        # lr_calculation in 'read' or 'retrieve'
        probs, pred_time, aux = lr_calculation(
            instance, memory, lr_exp=explainer,
            T_enc=T_enc, T_op=T_op, ddm_a=ddm_a, ddm_s=ddm_s,
            compute_sf=compute_sf,
            mode=("read" if with_xai else "retrieve"),
        )
        return probs, pred_time, aux

    elif strategy == "lr_heur":
        # heuristic maps T_enc->T_READ_NUM, T_op->T_INTUITIVE_OP
        probs, pred_time, info = lr_heuristic(
            norm_instance, memory, explainer,
            num_samples=40, K_top=3,
            T_READ_NUM=T_enc, T_INTUITIVE_OP=T_op,
            ddm_a=ddm_a, ddm_s=ddm_s, ddm_Tnd=0.30, ddm_norm="l2",
            active_indices=None, verbose=False
        )
        return probs, pred_time, info

    elif strategy == "dt":
        # dt_traverse in 'read' or 'retrieve'
        probs, pred_time, aux = dt_traverse(
            instance, memory, explainer,
            mode=("read" if with_xai else "retrieve"),
            compute_sf=int(compute_sf),
            T_enc=T_enc, ddm_a=ddm_a, ddm_s=ddm_s, ddm_Tnd=0.30, ddm_norm="l2",
            n_mc=64, topk_k=3, refresh_prob_cap=1.0, verbose=False
        )
        return probs, pred_time, aux

    else:
        raise ValueError(f"Unknown strategy: {strategy!r}")


def _post_read_refresh(strategy, memory, explainer, *, compute_sf, info_or_aux, instance, with_xai, actual_label):
    """
    Applies post-read/online refresh depending on strategy.
    - For lr_calc: refresh only when with_xai=True (as before).
    - For lr_heur: refresh on BOTH. If with_xai=False, use model’s own pred as 'actual'.
    - For dt:      after read, refresh the path; retrieve-mode may use refresh_prob_cap internally.
    """
    if strategy == "lr_calc":
        if with_xai:
            refresh_lr_calculation_in_memory(
                memory, explainer,
                intercept_display_sf=int(compute_sf),
                factor_display_sf=int(compute_sf),
            )

    elif strategy == "lr_heur":
        # 'info_or_aux' is the 'info' dict returned by lr_heuristic
        info = info_or_aux
        # actual_label already set by caller to either human response (with XAI) or model pred (w/o XAI)
        refresh_lr_heuristic_in_memory(
            memory, explainer, info, actual=int(actual_label),
            active_indices=None, w_min=1e-4, verbose=False
        )

    elif strategy == "dt":
        if with_xai:
            # refresh the path after reading explanations
            # (thresh_sf ties to compute_sf for consistency)
            refresh_dt_path_in_memory(memory, explainer, instance, thresh_sf=int(compute_sf))




# ---------------- Trial simulation (updated) ----------------
def _simulate_trials(
    T_enc, T_op, ddm_a, ddm_s, retrieval_threshold, latency_factor,
    forward_trials, ai_dataset_loader, explainer, model_name, app_id: str,
    compute_sf=2, lapse=0.0, *, strategy: str = "lr_calc"
):
    memory = _make_memory(retrieval_threshold, latency_factor)
    local_loader = filter_by_app_and_model(ai_dataset_loader, app_id, model_name)

    _seed_memory_for_strategy(strategy, memory, explainer, compute_sf)
    memory.tick(90)

    prob_list, rt_pred, rt_true = [], [], []
    trials = forward_trials.sort_values("Trial Index")

    for _, row in trials.iterrows():
        instance_id   = int(row["Instance Id"])
        actual_resp   = int(row["Response"])
        response_time = float(row["Time"])
        with_xai      = str(row.get("Tested w/ XAI", "w/o XAI")).strip().lower() in ("w/ xai","with xai","xai","1","true","yes")

        instances, preds = local_loader.load_instances([instance_id], normalize=False)
        instance = instances[0]

        norm_instance = local_loader.load_instances([instance_id], normalize=True)[0][0]

        # 1) forward pass
        probs, pred_time, aux = _predict_once(
            strategy, instance, norm_instance, memory, explainer,
            with_xai=with_xai,
            T_enc=T_enc, T_op=T_op, ddm_a=ddm_a, ddm_s=ddm_s,
            compute_sf=compute_sf
        )

        # 2) NLL with post-lapse mixing
        p = _select_prob(probs, actual_resp)
        if lapse > 0:
            p = (1.0 - lapse) * p + 0.5 * lapse

        # 3) Online update / refresh (strategy-specific)
        if strategy == "lr_heur":
            # If no XAI, use model’s own prediction as "actual"
            if not with_xai:
                label_for_update = preds[0]
            else:
                label_for_update = int(explainer.apply_to_instance(instance)>0)
            _post_read_refresh(
                "lr_heur", memory, explainer,
                compute_sf=compute_sf, info_or_aux=aux,
                instance=instance, with_xai=with_xai, actual_label=label_for_update
            )
        elif strategy == "lr_calc":
            _post_read_refresh(
                "lr_calc", memory, explainer,
                compute_sf=compute_sf, info_or_aux=aux,
                instance=instance, with_xai=with_xai, actual_label=None
            )
        elif strategy == "dt":
            _post_read_refresh(
                "dt", memory, explainer,
                compute_sf=compute_sf, info_or_aux=aux,
                instance=instance, with_xai=with_xai, actual_label=None
            )

        # 4) record metrics
        prob_list.append(float(p))
        rt_pred.append(float(pred_time))
        rt_true.append(response_time)

    return prob_list, rt_pred, rt_true


In [ ]:
# ===================== Action → Strategy mapping (outside the class) =====================

# 1–5 map onto: 2× LR-calculation (with/without XAI), 2× DT (with/without XAI), and 1× LR-heuristic (XAI follows trial flag)
STRATEGY_MAP = {
    1: {"strategy": "lr_calc", "with_xai": True},    # LR calculation, with XAI
    2: {"strategy": "lr_calc", "with_xai": False},   # LR calculation, without XAI
    3: {"strategy": "dt",       "with_xai": True},   # Decision Tree, with XAI
    4: {"strategy": "dt",       "with_xai": False},  # Decision Tree, without XAI
    5: {"strategy": "lr_heur",  "with_xai": None},   # LR heuristic; XAI will follow the trial's with_xai flag
}

def run_selected_strategy(
    action_id: int,
    *,
    instance_raw,         # numpy array (unnormalized instance)
    instance_norm,        # numpy array (normalized instance)
    memory,               # your CombinedMemory
    lr_exp=None,          # LogisticRegressionInterpreter (when needed)
    dt_exp=None,          # DecisionTreeInterpreter (when needed)
    with_xai_trial=False, # bool from the trial schedule
    active_indices=None,  # feature mask indices (list[int])
    # Strategy params (collected in the env from current_cog_params)
    T_enc=2.0, T_op=0.2, ddm_a=1.0, ddm_s=1.0, ddm_Tnd=0.30, ddm_norm="l2",
    compute_sf=2, lapse=0.0,
    num_samples=40, K_top=3,
):
    """
    Runs the chosen action via your existing _predict_once(strategy=..., ...) function.

    Returns: (probs, pred_time, aux_info)
    """
    spec = STRATEGY_MAP.get(int(action_id))
    if spec is None:
        raise ValueError(f"Unknown action id: {action_id}")

    strategy = spec["strategy"]
    with_xai = spec["with_xai"]
    # For lr_heuristic (action 5), let XAI follow the trial's schedule
    if with_xai is None:
        with_xai = bool(with_xai_trial)

    # Choose the right explainer for the strategy
    explainer = lr_exp if strategy in ("lr_calc", "lr_heur") else dt_exp

    # Delegate to your existing function (must be defined/imported elsewhere)
    probs, pred_time, aux = _predict_once(
        strategy,
        instance_raw,            # instance (unnormalized) for lr_calc/dt; your function also accepts norm_instance separately
        instance_norm,
        memory, explainer,
        with_xai=with_xai,
        T_enc=T_enc, T_op=T_op,
        ddm_a=ddm_a, ddm_s=ddm_s,
        compute_sf=compute_sf
    )
    return probs, float(pred_time), aux


# ============================== The RL Environment class ==============================

import gym
from gym import spaces
import numpy as np

class LRForward(gym.Env):
    """
    Minimal environment that chooses among 5 strategy actions + a feature mask.

    Actions:
      1: LR calculation  (with XAI)
      2: LR calculation  (without XAI)
      3: Decision Tree   (with XAI)
      4: Decision Tree   (without XAI)
      5: LR heuristic    (XAI follows the trial's with_xai flag)

    Observation (float32):
      [chi_norm, trial_idx_norm, with_xai_flag,
       strategy_counts[5], strategy_success_rates[5],
       per-feature contribution-std distribution [max_features],
       (optional) varied cognitive params...]

    Reward:
      prob_correct - chi * pred_time
    """

    metadata = {"render_modes": ["human"]}

    def __init__(self,
                 instances_per_episode: int = 40,
                 max_features: int = 6,
                 chi_low: float = 0.0,
                 chi_high: float = 0.03,
                 xai_trial_ratio: float = 0.5,
                 ai_dataset_loaders=None,     # dict[key] -> loader
                 lr_exps=None,                # dict[key] -> LR explainer
                 dt_exps=None,                # dict[key] -> DT explainer
                 training: bool = True,
                 cog_params: dict = None,     # dict of defaults or ranges (lo,hi)
                 condition: str = "Hybrid",   # "LR", "DT", or "Hybrid"
                 complexity: str = "low",     # "low" or "high"
                ):
        super().__init__()

        self.instances_per_episode = int(instances_per_episode)
        self.max_features = int(max_features)
        self.chi_low = float(chi_low)
        self.chi_high = float(chi_high)
        self.xai_trial_ratio = float(xai_trial_ratio)

        self.ai_dataset_loaders = ai_dataset_loaders or {}
        self.lr_exps = lr_exps or {}
        self.dt_exps = dt_exps or {}

        # Action = strategy id (1..5) + feature mask (binary per feature)
        self.action_space = spaces.MultiDiscrete([6] + [2] * self.max_features)  # 0..5, we will treat 0 as invalid (penalty)

        self.cog_params = cog_params or {}
        self.num_varied_cogparam = 0
        self.varied_cogparams_low = []
        self.varied_cogparams_high = []
        for k, v in self.cog_params.items():
            if isinstance(v, (list, tuple)) and len(v) == 2:
                self.num_varied_cogparam += 1
                self.varied_cogparams_low.append(float(v[0]))
                self.varied_cogparams_high.append(float(v[1]))

        # Observation space bounds
        low_vec = [0.0, 0.0, 0.0]                                  # chi_norm, trial_norm, with_xai_flag
        high_vec = [1.0, 1.0, 1.0]

        low_vec += [0.0] * 5                                       # strategy_counts (lo)
        high_vec += [float(self.instances_per_episode)] * 5        # strategy_counts (hi)

        low_vec += [0.0] * 5                                       # strategy_success rates (lo)
        high_vec += [1.0] * 5                                      # success rates (hi)

        low_vec += [0.0] * self.max_features                       # per-feature std-dist (lo)
        high_vec += [1.0] * self.max_features                      # per-feature std-dist (hi)

        if self.num_varied_cogparam > 0:
            low_vec += self.varied_cogparams_low
            high_vec += self.varied_cogparams_high

        self.observation_space = spaces.Box(
            low=np.array(low_vec, dtype=np.float32),
            high=np.array(high_vec, dtype=np.float32),
            dtype=np.float32,
        )

        self.step_idx = 0
        self.curr_chi = 0.0
        self.training = bool(training)

        # Track per-strategy usage and success (5 strategies now)
        self.strategy_counts = np.zeros(5, dtype=np.int32)
        self.strategy_success = np.zeros(5, dtype=np.float32)

        self.condition = condition
        self.complexity = complexity

        self.memory = None
        self.lr_exp = None
        self.dt_exp = None
        self.ai_dataset_loader = None

        self.contributions = {i: [] for i in range(self.max_features)}
        self.trials = None
        self.current_cog_params = {}

    # ---------------------- internal helpers ----------------------

    def _initialize_memory(self):
        memory_param_default = dict(
            retrieval_threshold=0.5,
            latency_factor=3.0,
            latency_exponent=0.5,
            max_assoc_strength=2.0,
            mismatch_penalty=-1.0,
            activation_noise=0.1,
            decay=0.5,
        )

        # override from current episode's cognitive params if provided
        for k, v in (self.current_cog_params or {}).items():
            if isinstance(v, (int, float)) and k in memory_param_default:
                memory_param_default[k] = float(v)

        dm = DeclarativeMemory(**memory_param_default)
        self.memory = CombinedMemory(dm, wm_capacity=10)

        # Optionally seed LR/DT knowledge
        if self.lr_exp is not None:
            add_lr_calculation_to_memory(self.lr_exp, self.memory)
            add_lr_heuristic_to_memory(self.lr_exp, self.memory, initial_sigma=0.1)
        if self.dt_exp is not None:
            root = self.dt_exp.tree_structure[0]
            add_dt_to_memory(self.memory, dt_exp=self.dt_exp, thresh_sf=2)

        self.memory.tick(90)

    def generate_episode_conditions(self):
        """
        Returns a list of per-trial dicts: {with_xai, model_type, complexity}.
        If self.condition is "LR" or "DT", all trials use that model_type.
        If "Hybrid", trials are half LR and half DT (shuffled).
        """
        rng = np.random.default_rng()
        n = self.instances_per_episode
        n_xai = round(n * self.xai_trial_ratio)

        flags = np.array([1] * n_xai + [0] * (n - n_xai))
        rng.shuffle(flags)

        if self.condition in ("LR", "DT"):
            trials = [dict(with_xai=bool(f), model_type=self.condition, complexity=self.complexity) for f in flags]
        else:
            n_lr = n // 2
            trials = (
                [dict(with_xai=bool(f), model_type="LR", complexity=self.complexity) for f in flags[:n_lr]] +
                [dict(with_xai=bool(f), model_type="DT", complexity=self.complexity) for f in flags[n_lr:]]
            )
            rng.shuffle(trials)
        return trials

    def _get_strategy_params(self):
        """
        Collect strategy-level knobs (timing & diffusion & display precision),
        allowing episode-varying overrides from self.current_cog_params.
        """
        params = dict(
            T_enc=2.0,
            T_op=0.2,
            ddm_a=1.0,
            ddm_s=1.0,
            ddm_Tnd=0.30,
            ddm_norm="l2",
            compute_sf=2,
            lapse=0.05,
            num_samples=40,
            K_top=3,
        )
        for k, v in (self.current_cog_params or {}).items():
            if k in params and isinstance(v, (int, float)):
                params[k] = float(v)
        return params

    def _build_obs(self):
        # guard
        if self.step_idx >= self.instances_per_episode:
            # return a valid zero-like obs if called after episode end
            zeros = np.zeros(self.observation_space.shape, dtype=np.float32)
            return zeros

        # With-XAI flag for current trial
        with_xai_flag = float(bool(self.trials[self.step_idx]["with_xai"]))

        # Feature contribution tracking (per-step update)
        x_raw = self.X_raw[self.step_idx]
        # Update contribution history per feature
        for feat in range(min(self.max_features, x_raw.shape[-1])):
            factor_value = self.lr_exp.coefficients.get(f"a{feat}", 0.0) if (self.lr_exp is not None and hasattr(self.lr_exp, "coefficients")) else 0.0
            self.contributions[feat].append(float(x_raw[feat]) * float(factor_value))

        # Compute per-feature std; normalize to sum=1 (if any non-zero)
        feature_stds = np.array([
            np.std(self.contributions[i]) if len(self.contributions[i]) > 1 else 0.0
            for i in range(self.max_features)
        ], dtype=np.float32)
        if feature_stds.sum() > 0:
            partial_sums_std = (feature_stds / feature_stds.sum()).astype(np.float32)
        else:
            partial_sums_std = feature_stds

        base = [
            float(self.curr_chi / max(self.chi_high, 1e-9)),
            float(self.step_idx / max(self.instances_per_episode, 1)),
            with_xai_flag,
            *self.strategy_counts.tolist(),
            *self.strategy_success.tolist(),
            *partial_sums_std.tolist(),
        ]

        if self.num_varied_cogparam > 0:
            cog_params_obs = [
                float(self.current_cog_params.get(k, 0.0))
                for k, v in self.cog_params.items()
                if isinstance(v, (list, tuple)) and len(v) == 2
            ]
            base += cog_params_obs

        return np.asarray(base, dtype=np.float32)

    # --------------------------- Gym API ---------------------------

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        rng = np.random.default_rng(seed)

        self.trials = self.generate_episode_conditions()

        # Randomly select a dataset key and bind loaders/experts
        if not self.ai_dataset_loaders:
            raise RuntimeError("ai_dataset_loaders is empty; provide at least one dataset loader.")
        dataset_key = rng.choice(list(self.ai_dataset_loaders.keys()))
        self.ai_dataset_loader = self.ai_dataset_loaders[dataset_key]
        self.lr_exp = (self.lr_exps or {}).get(dataset_key, None)
        self.dt_exp = (self.dt_exps or {}).get(dataset_key, None)

        # Sample/assign cognitive params for this episode
        self.current_cog_params = {}
        for k, v in (self.cog_params or {}).items():
            if isinstance(v, (list, tuple)) and len(v) == 2:
                self.current_cog_params[k] = float(rng.uniform(v[0], v[1]))
            elif isinstance(v, (int, float)):
                self.current_cog_params[k] = float(v)

        self.step_idx = 0
        self.curr_chi = float(rng.uniform(self.chi_low, self.chi_high))
        self._initialize_memory()

        # Reset trackers
        self.strategy_counts[:] = 0
        self.strategy_success[:] = 0.0
        self.contributions = {i: [] for i in range(self.max_features)}

        # Sample instances for the episode (adapt to your dataset size)
        all_idx = list(range(1, 400))
        idx = rng.choice(all_idx, size=self.instances_per_episode, replace=False).tolist()
        inst_raw, labels = self.ai_dataset_loader.load_instances(idx, normalize=False)
        inst_norm, _ = self.ai_dataset_loader.load_instances(idx, normalize=True)
        self.X_raw = np.asarray(inst_raw, dtype=np.float32)
        self.X_norm = np.asarray(inst_norm, dtype=np.float32)
        self.y = np.asarray(labels, dtype=np.int64)

        return self._build_obs(), {}

    def step(self, action):
        action = np.asarray(action, dtype=np.int64)
        action_id = int(action[0])
        mask_bits = action[1:].tolist()
        active_indices = [i for i, b in enumerate(mask_bits[:self.max_features]) if b == 1]

        # Hard-guard invalid action id 0 or >5
        if action_id not in STRATEGY_MAP:
            info = {"error": f"Invalid action id {action_id}", "chi": self.curr_chi}
            self.step_idx += 1
            truncated = self.step_idx >= self.instances_per_episode
            return self._build_obs(), -5.0, False, truncated, info

        # Current trial info
        trial = self.trials[self.step_idx]
        with_xai_trial = bool(trial["with_xai"])

        # Instances
        x_raw = self.X_raw[self.step_idx]
        x_norm = self.X_norm[self.step_idx]
        y_true = int(self.y[self.step_idx])

        # Get shared strategy params from episode-varying cognitive settings
        SP = self._get_strategy_params()

        # Run the chosen strategy (delegated outside the class)
        probs, pred_time, _ = run_selected_strategy(
            action_id,
            instance_raw=x_raw,
            instance_norm=x_norm,
            memory=self.memory,
            lr_exp=self.lr_exp,
            dt_exp=self.dt_exp,
            with_xai_trial=with_xai_trial,
            active_indices=active_indices,
            T_enc=SP["T_enc"], T_op=SP["T_op"],
            ddm_a=SP["ddm_a"], ddm_s=SP["ddm_s"],
            ddm_Tnd=SP["ddm_Tnd"], ddm_norm=SP["ddm_norm"],
            compute_sf=SP["compute_sf"], lapse=SP["lapse"],
            num_samples=int(SP["num_samples"]), K_top=int(SP["K_top"]),
        )

        # Reward: prob(correct) − chi × time   (optionally lapse-mix for reward too)
        prob_correct = float(probs[y_true])
        if SP["lapse"] > 0.0:
            prob_correct = (1.0 - SP["lapse"]) * prob_correct + 0.5 * SP["lapse"]
        reward = prob_correct - float(self.curr_chi) * float(pred_time)

        # Update per-strategy stats (index: 0..4 for actions 1..5)
        idx = action_id - 1
        self.strategy_counts[idx] += 1
        n = self.strategy_counts[idx]
        old_mean = self.strategy_success[idx]
        self.strategy_success[idx] = (old_mean * (n - 1) + (1.0 if prob_correct > 0.5 else 0.0)) / max(n, 1)

        # Advance time
        self.step_idx += 1
        truncated = self.step_idx >= self.instances_per_episode

        obs = self._build_obs()
        info = {
            "action_id": action_id,
            "with_xai_trial": with_xai_trial,
            "active_indices": active_indices,
            "prob_correct": prob_correct,
            "pred_time": float(pred_time),
            "chi": self.curr_chi,
            "cog_params": dict(self.current_cog_params),
        }
        return obs, float(reward), False, truncated, info


In [ ]:
# # ====== Gymnasium Env: LR-Calc Feature-Selection for Stable-Baselines ======
# # Requirements (import in your notebook/script):
# # pip install gymnasium numpy
# # (Stable-Baselines3 can interact with this env; it only requires standard Gymnasium API.)

# import math
# import numpy as np
# from collections import deque
# from typing import Dict, List, Optional, Tuple

# import gymnasium as gym
# from gymnasium import spaces

# # --- You must already have these imported from your codebase ---
# # from your_module import (
# #     DeclarativeMemory, CombinedMemory,
# #     lr_calculation, add_lr_calculation_to_memory, refresh_lr_calculation_in_memory,
# #     filter_by_app_and_model    # if you organize loaders like in cell 4 (optional)
# # )

# class LRCalcFeatureSelectEnv(gym.Env):
#     """
#     RL environment for **feature selection** using the LR calculation strategy only.

#     Action space:
#         - MultiBinary(max_features): a binary mask selecting which features are "active".

#     Observation space (vector):
#         [
#           0) chi_value (scalar, normalized: chi / chi_high)
#           1..8) cognitive parameters (8 scalars, in this order and normalized by their bounds):
#                 [T_enc, T_op, retrieval_threshold, latency_factor, ddm_a, ddm_s, lapse, compute_sf]
#           9) with_xai flag (0 or 1)
#           10) accumulated_accuracy (running mean correctness in [0,1])
#           11..15) past_correctness_history (5 entries, each 0 or 1; 0 when not filled yet)
#           16) last_reward (from the previous step; 0.0 at reset)
#           17) current_prediction (probability of the true class at this step)
#           18) prediction_time (seconds, model-produced)
#         ]

#     Episode flow:
#         - On each step, the agent selects a feature mask.
#         - The env runs `lr_calculation` with those active features:
#               mode="read" if with_xai else "retrieve"
#         - Reward = prob_correct - chi * pred_time   (consistent with cell 5 design)
#         - If with_xai=True, memory is refreshed via `refresh_lr_calculation_in_memory`.

#     Notes:
#         - The environment samples/schedules with-XAI trials according to `xai_trial_ratio`.
#         - Cognitive params are sampled uniformly within provided bounds unless fixed values are given.
#         - Dataset indices are sampled without replacement each episode.
#         - This env assumes:
#             * `ai_dataset_loader` exposes: load_instances(ids, normalize=False/True)
#               and optionally filter_by_app_and_model() if you pass app/model IDs.
#             * `explainer` is the LR explainer object used by lr_calculation.
#             * Memory helpers (DeclarativeMemory, CombinedMemory, add_lr_calculation_to_memory, refresh_lr_calculation_in_memory)
#               are available in the namespace (see your cell 4).
#     """

#     metadata = {"render_modes": ["human"]}

#     # ---- Bounds for cognitive parameters (from your spec) ----
#     BOUNDS = {
#         "T_enc": (0.05, 10.0),
#         "T_op": (0.05, 1.0),
#         "retrieval_threshold": (-2.0, -0.1),
#         "latency_factor": (0.3, 3.0),
#         "ddm_a": (0.5, 3.0),
#         "ddm_s": (0.5, 2.0),
#         "lapse": (0.0, 0.3),
#         "compute_sf": (1.0, 3.0),
#     }

#     COG_PARAM_ORDER = [
#         "T_enc", "T_op", "retrieval_threshold", "latency_factor",
#         "ddm_a", "ddm_s", "lapse", "compute_sf"
#     ]

#     def __init__(
#         self,
#         *,
#         instances_per_episode: int = 64,
#         max_features: int = 6,
#         chi_low: float = 0.0,
#         chi_high: float = 0.3,
#         xai_trial_ratio: float = 0.5,
#         ai_dataset_loader=None,   # required: must implement load_instances()
#         explainer=None,           # required: LR explainer used by lr_calculation
#         model_name: Optional[str] = None,   # optional, for loader filtering
#         app_id: Optional[str] = None,       # optional, for loader filtering
#         # Fix some cognitive params (dict) or leave None to sample within bounds.
#         fixed_cog_params: Optional[Dict[str, float]] = None,
#         # Reward scalars
#         time_penalty_scale: float = 0.1,    # final chi used is chi_value * time_penalty_scale
#         # Dataset indexing
#         instance_id_pool: Optional[List[int]] = None,  # default: 1..400
#         seed: Optional[int] = None
#     ):
#         super().__init__()

#         assert ai_dataset_loader is not None, "ai_dataset_loader is required."
#         assert explainer is not None, "explainer (LR explainer) is required."

#         self.rng = np.random.default_rng(seed)
#         self.instances_per_episode = int(instances_per_episode)
#         self.max_features = int(max_features)
#         self.chi_low = float(chi_low)
#         self.chi_high = float(chi_high)
#         self.xai_trial_ratio = float(xai_trial_ratio)
#         self.ai_dataset_loader = ai_dataset_loader
#         self.explainer = explainer
#         self.model_name = model_name
#         self.app_id = app_id
#         self.fixed_cog_params = fixed_cog_params or {}
#         self.time_penalty_scale = float(time_penalty_scale)
#         self.instance_id_pool = instance_id_pool if instance_id_pool is not None else list(range(1, 400))

#         # --- Action space: choose which features are active ---
#         self.action_space = spaces.MultiBinary(self.max_features)

#         # --- Observation space ---
#         # Low/High vectors must be concrete. We normalize bounded items to [0,1] where possible.
#         # chi (normalized), 8 cog params (normalized 0..1), with_xai, acc, 5 history, last_reward, prob, pred_time
#         obs_len = 1 + 8 + 1 + 1 + 5 + 1 + 1 + 1
#         low = np.zeros(obs_len, dtype=np.float32)
#         high = np.ones(obs_len, dtype=np.float32)
#         # last_reward can be negative; allow some range
#         low[16] = -1.0  # last_reward
#         high[16] = 1.0
#         # pred_time can be larger than 1; set a generous cap (e.g., 30s)
#         low[18] = 0.0
#         high[18] = 30.0
#         self.observation_space = spaces.Box(low=low, high=high, dtype=np.float32)

#         # --- Episode state ---
#         self.step_idx: int = 0
#         self.curr_chi: float = 0.0
#         self.with_xai_schedule: np.ndarray = np.zeros(self.instances_per_episode, dtype=np.int32)
#         self.memory = None

#         # Data buffers
#         self.X_raw = None
#         self.X_norm = None
#         self.y = None

#         # Stats
#         self.accumulated_accuracy = 0.0
#         self.correct_history = deque(maxlen=5)
#         self.last_reward = 0.0
#         self.last_prob_correct = 0.0
#         self.last_pred_time = 0.0

#         # Current cognitive params (dict)
#         self.cog_params: Dict[str, float] = {}

#     # ---------- Helpers ----------
#     @staticmethod
#     def _normalize(value: float, lo: float, hi: float) -> float:
#         return float((value - lo) / (hi - lo + 1e-12))

#     def _normalize_cogs(self, params: Dict[str, float]) -> List[float]:
#         vals = []
#         for k in self.COG_PARAM_ORDER:
#             lo, hi = self.BOUNDS[k]
#             vals.append(self._normalize(params[k], lo, hi))
#         return vals

#     def _initialize_memory(self):
#         """
#         Build a DeclarativeMemory + CombinedMemory consistent with cell 4,
#         using *current* cognitive parameters for memory-related fields.
#         """
#         # memory params derived from cognitive params
#         dm = DeclarativeMemory(
#             retrieval_threshold=self.cog_params["retrieval_threshold"],
#             latency_factor=self.cog_params["latency_factor"],
#             latency_exponent=0.5,
#             max_assoc_strength=2.0,
#             mismatch_penalty=-1.0,
#             activation_noise=0.3,
#             decay=0.5,
#         )
#         self.memory = CombinedMemory(dm, wm_capacity=7)
#         # Preload LR coefficients/state for calculation path
#         add_lr_calculation_to_memory(self.explainer, self.memory)
#         self.memory.tick(10)

#     def _build_obs(self, with_xai_flag: int) -> np.ndarray:
#         # chi normalized
#         chi_norm = self.curr_chi / max(self.chi_high, 1e-9)

#         # normalized cogs
#         cog_norm = self._normalize_cogs(self.cog_params)

#         # pad history to length 5 for early steps
#         hist = list(self.correct_history)
#         if len(hist) < 5:
#             hist = hist + [0.0] * (5 - len(hist))
#         hist = hist[:5]

#         obs = np.array(
#             [chi_norm] +
#             cog_norm +
#             [float(with_xai_flag)] +
#             [float(self.accumulated_accuracy)] +
#             [float(h) for h in hist] +
#             [float(self.last_reward)] +
#             [float(self.last_prob_correct)] +
#             [float(self.last_pred_time)],
#             dtype=np.float32
#         )
#         return obs

#     def _sample_cog_params(self) -> Dict[str, float]:
#         params = {}
#         for k in self.COG_PARAM_ORDER:
#             if k in self.fixed_cog_params and isinstance(self.fixed_cog_params[k], (int, float)):
#                 params[k] = float(self.fixed_cog_params[k])
#             else:
#                 lo, hi = self.BOUNDS[k]
#                 params[k] = float(self.rng.uniform(lo, hi))
#         # compute_sf is discrete display SF in {1,2,3}; sample continuous then snap
#         cs_lo, cs_hi = self.BOUNDS["compute_sf"]
#         params["compute_sf"] = int(np.clip(round(params["compute_sf"]), cs_lo, cs_hi))
#         return params

#     # ---------- Gymnasium API ----------
#     def reset(self, *, seed: Optional[int] = None, options: Optional[dict] = None):
#         if seed is not None:
#             self.rng = np.random.default_rng(seed)

#         # Schedule with-XAI flags
#         n = self.instances_per_episode
#         n_xai = int(round(n * self.xai_trial_ratio))
#         flags = np.array([1] * n_xai + [0] * (n - n_xai), dtype=np.int32)
#         self.rng.shuffle(flags)
#         self.with_xai_schedule = flags

#         # Cognitive params
#         self.cog_params = self._sample_cog_params()

#         # Initialize memory
#         self._initialize_memory()

#         # Draw chi and instances
#         self.curr_chi = float(self.rng.uniform(self.chi_low, self.chi_high))

#         idx = self.rng.choice(self.instance_id_pool, size=self.instances_per_episode, replace=False).tolist()
#         inst_raw, labels = self.ai_dataset_loader.load_instances(idx, normalize=False)
#         inst_norm, _ = self.ai_dataset_loader.load_instances(idx, normalize=True)

#         self.X_raw = np.asarray(inst_raw, dtype=np.float32)
#         self.X_norm = np.asarray(inst_norm, dtype=np.float32)
#         self.y = np.asarray(labels, dtype=np.int64)

#         # Reset stats
#         self.step_idx = 0
#         self.accumulated_accuracy = 0.0
#         self.correct_history.clear()
#         self.last_reward = 0.0
#         self.last_prob_correct = 0.0
#         self.last_pred_time = 0.0

#         obs = self._build_obs(with_xai_flag=int(self.with_xai_schedule[0]))
#         info = {"cog_params": self.cog_params.copy(), "chi": self.curr_chi}
#         return obs, info

#     def step(self, action):
#         # Validate and parse action -> active feature indices
#         action = np.asarray(action, dtype=np.int64).flatten()
#         if action.size != self.max_features:
#             raise ValueError(f"Expected action of size {self.max_features}, got {action.size}")
#         active_indices = [i for i, b in enumerate(action.tolist()) if b == 1]

#         # End if already past episode length
#         terminated = False
#         truncated = self.step_idx >= self.instances_per_episode
#         if truncated:
#             return self._build_obs(with_xai_flag=0), 0.0, terminated, truncated, {}

#         # Fetch trial info
#         with_xai = bool(self.with_xai_schedule[self.step_idx])

#         # Current instance
#         x_raw = self.X_raw[self.step_idx]
#         y_true = int(self.y[self.step_idx])

#         # Map required cog params
#         T_enc = float(self.cog_params["T_enc"])
#         T_op = float(self.cog_params["T_op"])
#         ddm_a = float(self.cog_params["ddm_a"])
#         ddm_s = float(self.cog_params["ddm_s"])
#         compute_sf = int(self.cog_params["compute_sf"])
#         lapse = float(self.cog_params["lapse"])  # optional post-mix

#         # Run LR calculation with the selected features
#         # In your lr_calculation signature, pass 'active_indices' if supported; else, handle masking inside.
#         probs, pred_time, aux = lr_calculation(
#             x_raw, self.memory, lr_exp=self.explainer,
#             T_enc=T_enc, T_op=T_op, ddm_a=ddm_a, ddm_s=ddm_s,
#             compute_sf=compute_sf,
#             mode=("read" if with_xai else "retrieve"),
#             # active_indices=active_indices,   # uncomment if your lr_calculation supports it
#         )

#         # Select probability of the TRUE label (assuming probs=[p0,p1])
#         p_true = float(probs[y_true])
#         if lapse > 0.0:
#             p_true = (1.0 - lapse) * p_true + 0.5 * lapse

#         # Reward: accuracy term minus time cost scaled by chi (like cell 5)
#         reward = p_true - (self.curr_chi * self.time_penalty_scale) * float(pred_time)

#         # If read-with-XAI, refresh memory (as specified)
#         if with_xai:
#             refresh_lr_calculation_in_memory(
#                 self.memory, self.explainer,
#                 intercept_display_sf=int(compute_sf),
#                 factor_display_sf=int(compute_sf),
#             )

#         # Update stats
#         correct = 1.0 if p_true > 0.5 else 0.0
#         self.correct_history.append(correct)
#         self.accumulated_accuracy = (
#             (self.accumulated_accuracy * self.step_idx + correct) / (self.step_idx + 1)
#         )

#         # Advance
#         self.step_idx += 1
#         truncated = self.step_idx >= self.instances_per_episode

#         # Save lasts
#         self.last_reward = float(reward)
#         self.last_prob_correct = float(p_true)
#         self.last_pred_time = float(pred_time)

#         # Build next obs (with next with_xai flag if available)
#         next_with_xai_flag = int(self.with_xai_schedule[self.step_idx - 1] if truncated else self.with_xai_schedule[self.step_idx])
#         obs = self._build_obs(with_xai_flag=next_with_xai_flag)

#         info = {
#             "active_indices": active_indices,
#             "with_xai": with_xai,
#             "prob_correct": float(p_true),
#             "pred_time": float(pred_time),
#             "y_true": y_true,
#             "cog_params": self.cog_params.copy(),
#             "chi": self.curr_chi,
#         }
#         return obs, float(reward), terminated, truncated, info

#     # Optional: Stable-Baselines may call render/close
#     def render(self):
#         pass

#     def close(self):
#         pass


In [ ]:
# # ====== Gymnasium Env: LR-Calc Feature-Selection
# # —— dataset- & complexity-specific retrievers for (loader, explainer) pairs ——
# # Notes:
# # • Action = MultiBinary(max_features) (feature mask)
# # • Observation includes the *past* feature mask (length=max_features)
# # • At reset(), the env picks a dataset_id (random or via options["dataset_id"])
# #   and a complexity ∈ {"low","high"}; it then retrieves the matching (loader, explainer).
# # • Only the LR calculation strategy is used; memory refresh happens only when with_xai=True.

# import numpy as np
# from collections import deque
# from typing import Dict, List, Optional, Any

# import gymnasium as gym
# from gymnasium import spaces

# # --- You must provide these from your codebase ---
# # from your_module import (
# #     DeclarativeMemory, CombinedMemory,
# #     lr_calculation, add_lr_calculation_to_memory, refresh_lr_calculation_in_memory
# # )

# class LRCalcFeatureSelectEnv(gym.Env):
#     """
#     Constructor expects `datasets` shaped like:
#         datasets = {
#             "<dataset_id_A>": {
#                 "low":  {"loader": <LoaderA_low>,  "explainer": <ExplainerA_low>},
#                 "high": {"loader": <LoaderA_high>, "explainer": <ExplainerA_high>},
#             },
#             "<dataset_id_B>": {
#                 "low":  {"loader": <LoaderB_low>,  "explainer": <ExplainerB_low>},
#                 "high": {"loader": <LoaderB_high>, "explainer": <ExplainerB_high>},
#             },
#             ...
#         }

#     Observation vector (order):
#       [ chi_norm,
#         8 * normalized cognitive params
#           (T_enc, T_op, retrieval_threshold, latency_factor, ddm_a, ddm_s, lapse, compute_sf),
#         with_xai_flag,
#         accumulated_accuracy,
#         5 * past_correctness_history,
#         last_reward,
#         last_prob_correct,
#         last_pred_time,
#         max_features * past_feature_mask
#       ]

#     Reward:
#         r = p_true - (chi * time_penalty_scale) * pred_time
#     """

#     metadata = {"render_modes": ["human"]}

#     # Bounds per spec
#     BOUNDS = {
#         "T_enc": (0.05, 10.0),
#         "T_op": (0.05, 1.0),
#         "retrieval_threshold": (-2.0, -0.1),
#         "latency_factor": (0.3, 3.0),
#         "ddm_a": (0.5, 3.0),
#         "ddm_s": (0.5, 2.0),
#         "lapse": (0.0, 0.3),
#         "compute_sf": (1.0, 3.0),
#     }
#     COG_PARAM_ORDER = [
#         "T_enc", "T_op", "retrieval_threshold", "latency_factor",
#         "ddm_a", "ddm_s", "lapse", "compute_sf"
#     ]

#     def __init__(
#         self,
#         *,
#         datasets: Dict[str, Dict[str, Dict[str, Any]]],  # {dataset_id: {"low":{"loader","explainer"},"high":{...}}}
#         instances_per_episode: int = 64,
#         max_features: int = 6,
#         chi_low: float = 0.0,
#         chi_high: float = 0.03,
#         xai_trial_ratio: float = 0.5,
#         complexity_high_ratio: float = 0.5,   # P(episode complexity = "high")
#         fixed_cog_params: Optional[Dict[str, float]] = None,
#         time_penalty_scale: float = 1.0,
#         instance_id_pool: Optional[List[int]] = None,
#         seed: Optional[int] = None,
#     ):
#         super().__init__()
#         if not datasets:
#             raise ValueError("`datasets` mapping is required and cannot be empty.")
#         self.datasets = datasets
#         self.dataset_ids = list(datasets.keys())

#         self.rng = np.random.default_rng(seed)
#         self.instances_per_episode = int(instances_per_episode)
#         self.max_features = int(max_features)
#         self.chi_low, self.chi_high = float(chi_low), float(chi_high)
#         self.xai_trial_ratio = float(xai_trial_ratio)
#         self.complexity_high_ratio = float(np.clip(complexity_high_ratio, 0.0, 1.0))
#         self.fixed_cog_params = fixed_cog_params or {}
#         self.time_penalty_scale = float(time_penalty_scale)
#         self.instance_id_pool = instance_id_pool if instance_id_pool is not None else list(range(1, 400))

#         # Selected per episode
#         self.dataset_id: Optional[str] = None
#         self.episode_complexity: str = "low"    # "low" | "high"
#         self.ai_dataset_loader = None
#         self.explainer = None

#         # Action: choose active features
#         self.action_space = spaces.MultiBinary(self.max_features)

#         # Observation space (see class docstring)
#         len_base = 1                 # chi_norm
#         len_cogs = 8                 # normalized cogs
#         len_flags = 1                # with_xai flag
#         len_acc = 1                  # accumulated_accuracy
#         len_hist = 5                 # correctness history
#         len_tail = 3                 # last_reward, last_prob_correct, last_pred_time
#         len_mask = self.max_features # past feature-selection mask
#         obs_len = len_base + len_cogs + len_flags + len_acc + len_hist + len_tail + len_mask

#         low = np.zeros(obs_len, dtype=np.float32)
#         high = np.ones(obs_len, dtype=np.float32)
#         # last_reward (start of tail)
#         last_reward_idx = len_base + len_cogs + len_flags + len_acc + len_hist
#         low[last_reward_idx] = -1.0
#         high[last_reward_idx] = 1.0
#         # last_pred_time (tail index +2)
#         last_pred_time_idx = last_reward_idx + 2
#         low[last_pred_time_idx] = 0.0
#         high[last_pred_time_idx] = 30.0

#         self.observation_space = spaces.Box(low=low, high=high, dtype=np.float32)

#         # Episode state
#         self.step_idx = 0
#         self.curr_chi = 0.0
#         self.with_xai_schedule = np.zeros(self.instances_per_episode, dtype=np.int32)
#         self.memory = None

#         # Data buffers
#         self.X_raw = None
#         self.X_norm = None
#         self.y = None

#         # Stats
#         self.accumulated_accuracy = 0.0
#         self.correct_history = deque(maxlen=len_hist)
#         self.last_reward = 0.0
#         self.last_prob_correct = 0.0
#         self.last_pred_time = 0.0
#         self.last_feature_mask = np.zeros(self.max_features, dtype=np.float32)

#         # Current cognitive params
#         self.cog_params: Dict[str, float] = {}

#     # ---------------- Helpers ----------------
#     @staticmethod
#     def _normalize(v: float, lo: float, hi: float) -> float:
#         return float((v - lo) / (hi - lo + 1e-12))

#     def _normalize_cogs(self, params: Dict[str, float]) -> List[float]:
#         return [self._normalize(params[k], *self.BOUNDS[k]) for k in self.COG_PARAM_ORDER]

#     def _choose_complexity(self) -> str:
#         return "high" if self.rng.random() < self.complexity_high_ratio else "low"

#     def _pick_dataset_and_bind(self, *, dataset_id: Optional[str], complexity: str):
#         """
#         Choose a dataset_id (random if None) and bind loader/explainer
#         for the requested complexity ("low" or "high").
#         """
#         if dataset_id is None:
#             dataset_id = self.rng.choice(self.dataset_ids).item() if hasattr(self.rng.choice(self.dataset_ids), "item") else self.rng.choice(self.dataset_ids)
#         if dataset_id not in self.datasets:
#             raise KeyError(f"dataset_id '{dataset_id}' not found in provided datasets.")
#         if complexity not in self.datasets[dataset_id]:
#             raise KeyError(f"complexity '{complexity}' not available for dataset_id '{dataset_id}'.")

#         entry = self.datasets[dataset_id][complexity]
#         if not isinstance(entry, dict) or "loader" not in entry or "explainer" not in entry:
#             raise ValueError(f"datasets['{dataset_id}']['{complexity}'] must be a dict with 'loader' and 'explainer'.")

#         self.dataset_id = dataset_id
#         self.ai_dataset_loader = entry["loader"]
#         self.explainer = entry["explainer"]

#     def _initialize_memory(self):
#         dm = DeclarativeMemory(
#             retrieval_threshold=self.cog_params["retrieval_threshold"],
#             latency_factor=self.cog_params["latency_factor"],
#             latency_exponent=0.5,
#             max_assoc_strength=2.0,
#             mismatch_penalty=-1.0,
#             activation_noise=0.3,
#             decay=0.5,
#         )
#         self.memory = CombinedMemory(dm, wm_capacity=7)
#         add_lr_calculation_to_memory(self.explainer, self.memory)
#         self.memory.tick(10)

#     def _build_obs(self, with_xai_flag: int) -> np.ndarray:
#         chi_norm = self.curr_chi / max(self.chi_high, 1e-9)
#         cog_norm = self._normalize_cogs(self.cog_params)
#         hist = list(self.correct_history)
#         if len(hist) < self.correct_history.maxlen:
#             hist += [0.0] * (self.correct_history.maxlen - len(hist))

#         obs = np.array(
#             [chi_norm] +
#             cog_norm +
#             [float(with_xai_flag)] +
#             [float(self.accumulated_accuracy)] +
#             [float(h) for h in hist] +
#             [float(self.last_reward), float(self.last_prob_correct), float(self.last_pred_time)],
#             dtype=np.float32
#         )
#         obs = np.concatenate([obs, self.last_feature_mask.astype(np.float32)], dtype=np.float32)
#         return obs

#     def _sample_cog_params(self) -> Dict[str, float]:
#         params = {}
#         for k in self.COG_PARAM_ORDER:
#             if k in self.fixed_cog_params and isinstance(self.fixed_cog_params[k], (int, float)):
#                 params[k] = float(self.fixed_cog_params[k])
#             else:
#                 lo, hi = self.BOUNDS[k]
#                 params[k] = float(self.rng.uniform(lo, hi))
#         # Snap compute_sf to {1,2,3}
#         lo_cs, hi_cs = self.BOUNDS["compute_sf"]
#         params["compute_sf"] = int(np.clip(round(params["compute_sf"]), lo_cs, hi_cs))
#         return params

#     # ---------------- Gymnasium API ----------------
#     def reset(self, *, seed: Optional[int] = None, options: Optional[dict] = None):
#         if seed is not None:
#             self.rng = np.random.default_rng(seed)

#         # 1) Choose complexity and dataset_id (respect options if provided)
#         self.episode_complexity = self._choose_complexity()
#         ds_id_opt = None
#         if options and isinstance(options, dict) and "dataset_id" in options:
#             ds_id_opt = str(options["dataset_id"])
#         self._pick_dataset_and_bind(dataset_id=ds_id_opt, complexity=self.episode_complexity)

#         # 2) Build with-XAI schedule
#         n = self.instances_per_episode
#         n_xai = int(round(n * self.xai_trial_ratio))
#         flags = np.array([1] * n_xai + [0] * (n - n_xai), dtype=np.int32)
#         self.rng.shuffle(flags)
#         self.with_xai_schedule = flags

#         # 3) Cognitive params & memory
#         self.cog_params = self._sample_cog_params()
#         self._initialize_memory()

#         # 4) Episode chi
#         self.curr_chi = float(self.rng.uniform(self.chi_low, self.chi_high))

#         # 5) Data for this episode from the chosen loader
#         idx = self.rng.choice(self.instance_id_pool, size=self.instances_per_episode, replace=False).tolist()
#         inst_raw, labels = self.ai_dataset_loader.load_instances(idx, normalize=False)
#         inst_norm, _ = self.ai_dataset_loader.load_instances(idx, normalize=True)
#         self.X_raw = np.asarray(inst_raw, dtype=np.float32)
#         self.X_norm = np.asarray(inst_norm, dtype=np.float32)  # kept if you later need normalized
#         self.y = np.asarray(labels, dtype=np.int64)

#         # 6) Reset stats
#         self.step_idx = 0
#         self.accumulated_accuracy = 0.0
#         self.correct_history.clear()
#         self.last_reward = 0.0
#         self.last_prob_correct = 0.0
#         self.last_pred_time = 0.0
#         self.last_feature_mask = np.zeros(self.max_features, dtype=np.float32)

#         obs = self._build_obs(with_xai_flag=int(self.with_xai_schedule[0]))
#         info = {
#             "dataset_id": self.dataset_id,
#             "episode_complexity": self.episode_complexity,
#             "cog_params": self.cog_params.copy(),
#             "chi": self.curr_chi,
#         }
#         return obs, info

#     def step(self, action):
#         action = np.asarray(action, dtype=np.int64).flatten()
#         if action.size != self.max_features:
#             raise ValueError(f"Expected action of size {self.max_features}, got {action.size}")
#         active_indices = [i for i, b in enumerate(action.tolist()) if b == 1]

#         terminated = False
#         truncated = self.step_idx >= self.instances_per_episode
#         if truncated:
#             return self._build_obs(with_xai_flag=0), 0.0, terminated, truncated, {}

#         with_xai = bool(self.with_xai_schedule[self.step_idx])

#         x_raw = self.X_raw[self.step_idx]
#         y_true = int(self.y[self.step_idx])

#         T_enc = float(self.cog_params["T_enc"])
#         T_op = float(self.cog_params["T_op"])
#         ddm_a = float(self.cog_params["ddm_a"])
#         ddm_s = float(self.cog_params["ddm_s"])
#         compute_sf = int(self.cog_params["compute_sf"])
#         lapse = float(self.cog_params["lapse"])

#         # Run LR calculation (mask inside lr_calculation if supported)
#         probs, pred_time, _ = lr_calculation(
#             x_raw, self.memory, lr_exp=self.explainer,
#             T_enc=T_enc, T_op=T_op, ddm_a=ddm_a, ddm_s=ddm_s,
#             compute_sf=compute_sf,
#             mode=("read" if with_xai else "retrieve"),
#             # active_indices=active_indices,  # uncomment if your lr_calculation supports feature masking
#         )

#         p_true = float(probs[y_true])
#         if lapse > 0.0:
#             p_true = (1.0 - lapse) * p_true + 0.5 * lapse

#         reward = p_true - (self.curr_chi * self.time_penalty_scale) * float(pred_time)

#         if with_xai:
#             refresh_lr_calculation_in_memory(
#                 self.memory, self.explainer,
#                 intercept_display_sf=int(compute_sf),
#                 factor_display_sf=int(compute_sf),
#             )

#         correct = 1.0 if p_true > 0.5 else 0.0
#         self.correct_history.append(correct)
#         self.accumulated_accuracy = (
#             (self.accumulated_accuracy * self.step_idx + correct) / (self.step_idx + 1)
#         )

#         self.step_idx += 1
#         truncated = self.step_idx >= self.instances_per_episode

#         # Update past feature mask for NEXT observation
#         self.last_feature_mask = action.astype(np.float32)

#         self.last_reward = float(reward)
#         self.last_prob_correct = float(p_true)
#         self.last_pred_time = float(pred_time)

#         next_with_xai_flag = int(self.with_xai_schedule[self.step_idx - 1] if truncated
#                                  else self.with_xai_schedule[self.step_idx])
#         obs = self._build_obs(with_xai_flag=next_with_xai_flag)

#         info = {
#             "dataset_id": self.dataset_id,
#             "episode_complexity": self.episode_complexity,
#             "active_indices": active_indices,
#             "with_xai": with_xai,
#             "prob_correct": float(p_true),
#             "pred_time": float(pred_time),
#             "y_true": y_true,
#             "cog_params": self.cog_params.copy(),
#             "chi": self.curr_chi,
#         }
#         return obs, float(reward), terminated, truncated, info

#     def render(self): pass
#     def close(self): pass
